In [7]:
import functions
import importlib
importlib.reload(functions)

from functions import *
import matplotlib.pyplot as plt


In [8]:
import pandas as pd
import numpy as np

C_MPS = 299_792_458.0

# load full extracted GPS observations
df = pd.read_csv("GPS_C1C_D1C.csv")
df["time"] = pd.to_datetime(df["time"])

# exact satellites and time window
sv_order = ["G03", "G06", "G09", "G16", "G21"]
start_time = pd.Timestamp("2026-01-11 02:00:00")
end_time = start_time + pd.Timedelta(minutes=12.5)

# slice the window
window_df = df[
    (df["sv"].isin(sv_order)) &
    (df["time"] >= start_time) &
    (df["time"] <= end_time)
].copy()

window_df = window_df.sort_values(["time", "sv"]).reset_index(drop=True)
window_df["delay_s"] = window_df["C1C_m"] / C_MPS

print("rows:", len(window_df))
print("time min/max:", window_df["time"].min(), window_df["time"].max())
print("sats present:", window_df["sv"].unique())

# one dataframe per satellite
sat_frames = {
    sv: grp.copy().reset_index(drop=True)
    for sv, grp in window_df.groupby("sv")
}

# make sure all requested sats are present
for sv in sv_order:
    if sv not in sat_frames:
        print(f"Missing satellite: {sv}")

# build sat_data dict in the format you were already using
sat_data = {
    sv: {
        "time": sat_frames[sv]["time"].tolist(),
        "pseudorange_m": sat_frames[sv]["C1C_m"].tolist(),
        "delay_s": sat_frames[sv]["delay_s"].tolist(),
        "doppler_hz": sat_frames[sv]["D1C_hz"].tolist(),
    }
    for sv in sv_order
    if sv in sat_frames
}

# align into matrices with one common time index
pivot_pr = window_df.pivot(index="time", columns="sv", values="C1C_m").reindex(columns=sv_order)
pivot_delay = window_df.pivot(index="time", columns="sv", values="delay_s").reindex(columns=sv_order)
pivot_dopp = window_df.pivot(index="time", columns="sv", values="D1C_hz").reindex(columns=sv_order)

# interpolate missing values inside the window
pivot_pr = pivot_pr.interpolate(limit_direction="both")
pivot_delay = pivot_delay.interpolate(limit_direction="both")
pivot_dopp = pivot_dopp.interpolate(limit_direction="both")

# arrays for simulator: shape = (n_sats, n_times)
pseudorange_list = pivot_pr.to_numpy(dtype=float).T
delay_list = pivot_delay.to_numpy(dtype=float).T
doppler_list = pivot_dopp.to_numpy(dtype=float).T
time_list = pivot_delay.index.to_numpy()

print("pseudorange_list shape:", pseudorange_list.shape)
print("delay_list shape:", delay_list.shape)
print("doppler_list shape:", doppler_list.shape)
print("time_list shape:", time_list.shape)

# optional: show first few values for each sat
for i, sv in enumerate(sv_order):
    print(f"\n{sv}")
    print(" first times:", time_list[:3])
    print(" first pseudoranges:", pseudorange_list[i, :3])
    print(" first delays:", delay_list[i, :3])
    print(" first dopplers:", doppler_list[i, :3])

rows: 3755
time min/max: 2026-01-11 02:00:00 2026-01-11 02:12:30
sats present: <StringArray>
['G03', 'G06', 'G09', 'G16', 'G21']
Length: 5, dtype: str
pseudorange_list shape: (5, 751)
delay_list shape: (5, 751)
doppler_list shape: (5, 751)
time_list shape: (751,)

G03
 first times: ['2026-01-11T02:00:00.000000' '2026-01-11T02:00:01.000000'
 '2026-01-11T02:00:02.000000']
 first pseudoranges: [22550966.035 22551578.624 22552191.166]
 first delays: [0.07522193 0.07522397 0.07522601]
 first dopplers: [-3219.178 -3218.826 -3219.12 ]

G06
 first times: ['2026-01-11T02:00:00.000000' '2026-01-11T02:00:01.000000'
 '2026-01-11T02:00:02.000000']
 first pseudoranges: [21282939.357 21283012.485 21283085.85 ]
 first delays: [0.07099224 0.07099249 0.07099273]
 first dopplers: [-384.508 -384.57  -385.336]

G09
 first times: ['2026-01-11T02:00:00.000000' '2026-01-11T02:00:01.000000'
 '2026-01-11T02:00:02.000000']
 first pseudoranges: [20109800.981 20109624.281 20109447.688]
 first delays: [0.06707908 0

In [ ]:
#WITHOUT VARIABLE DOPPLER
C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_iq_file(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz=None,
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=10_000_000,
):
    """
    Multi-satellite complex baseband IQ generator.

    Writes interleaved float32 IQ samples:
        I0, Q0, I1, Q1, I2, Q2, ...

    Parameters
    ----------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite.
    distances_m : array-like
        One distance per satellite [m].
    out_file : str
        Output filename.
    chip_rate : float
        Chip rate [Hz].
    fs : float
        Sample rate [Hz].
    doppler_hz : array-like or None
        Per-satellite Doppler [Hz]. Defaults to zeros.
    phase0 : array-like or None
        Per-satellite initial carrier phase [rad]. Defaults to zeros.
    amplitudes : array-like or None
        Per-satellite amplitudes. Defaults to ones.
    use_relative_delays : bool
        If True, subtract the minimum geometric delay so the earliest
        satellite starts at t=0, then add common_delay_s.
    common_delay_s : float
        Extra delay added to all satellites. Useful if you want all
        pseudoranges shifted upward by a common amount.
    chunk_size : int
        Samples per write chunk.

    Returns
    -------
    n_total_samples : int
        Total number of complex samples written.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    n_sats = len(chip_streams_01)

    if len(distances_m) != n_sats:
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams from 0/1 to -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    n_signal_samples = int(np.floor(signal_duration_s * fs))

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    delays_s = delays_s + float(common_delay_s)
    delay_samples = np.round(delays_s * fs).astype(np.int64)

    if doppler_hz is None:
        doppler_hz = np.zeros(n_sats, dtype=np.float64)
    else:
        doppler_hz = np.asarray(doppler_hz, dtype=np.float64)
        if len(doppler_hz) != n_sats:
            raise ValueError("doppler_hz must have same length as chip_streams_01")

    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    max_delay_samples = int(np.max(delay_samples))
    n_total_samples = n_signal_samples + max_delay_samples

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Doppler [Hz]:", doppler_hz)
    print("Signal samples (undelayed):", n_signal_samples)
    print("Total complex samples:", n_total_samples)

    with open(out_file, "wb") as f:
        for start in range(0, n_total_samples, chunk_size):
            stop = min(start + chunk_size, n_total_samples)
            sample_idx = np.arange(start, stop, dtype=np.int64)

            i_sum = np.zeros(stop - start, dtype=np.float32)
            q_sum = np.zeros(stop - start, dtype=np.float32)

            for s in range(n_sats):
                src_sample_idx = sample_idx - delay_samples[s]
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

                if not np.any(valid):
                    continue

                # Sample the chip stream
                chip_index = np.floor(
                    src_sample_idx[valid].astype(np.float64) * chip_rate / fs
                ).astype(np.int64)
                chip_index = np.clip(chip_index, 0, n_chips - 1)
                baseband = chips[s][chip_index].astype(np.float32)

                # Complex baseband carrier with Doppler only
                t = sample_idx[valid].astype(np.float64) / fs
                phase = 2.0 * np.pi * doppler_hz[s] * t + phase0[s]

                i_sum[valid] += amplitudes[s] * baseband * np.cos(phase).astype(np.float32)
                q_sum[valid] += amplitudes[s] * baseband * np.sin(phase).astype(np.float32)

            iq = np.empty(2 * (stop - start), dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

    return n_total_samples

In [12]:
#WITH VARIABLE DOPPLER

C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz_chunks=None,
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=37500*20*1023/375,
):
    """
    Multi-satellite complex baseband IQ generator with chunk-varying Doppler.

    Writes interleaved float32 IQ samples:
        I0, Q0, I1, Q1, I2, Q2, ...

    Parameters
    ----------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite.
    distances_m : array-like
        One distance per satellite [m].
    out_file : str
        Output filename.
    chip_rate : float
        Chip rate [Hz].
    fs : float
        Sample rate [Hz].
    doppler_hz_chunks : array-like or None
        Doppler schedule per chunk, shape:
            (n_sats, n_chunks)
        or
            (n_chunks, n_sats)
        If None, all Dopplers are zero.
    phase0 : array-like or None
        Per-satellite initial carrier phase [rad]. Defaults to zeros.
    amplitudes : array-like or None
        Per-satellite amplitudes. Defaults to ones.
    use_relative_delays : bool
        If True, subtract the minimum geometric delay so the earliest
        satellite starts at t=0, then add common_delay_s.
    common_delay_s : float
        Extra delay added to all satellites.
    chunk_size : int
        Samples per write chunk.

    Returns
    -------
    n_total_samples : int
        Total number of complex samples written.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    n_sats = len(chip_streams_01)

    if len(distances_m) != n_sats:
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams from 0/1 to -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))
    print(chips,"chips")
    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")
    print(lengths,"length")
    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    print(signal_duration_s,"signal duration")
    n_signal_samples = int(np.floor(signal_duration_s * fs))
    print(n_signal_samples,"n total samples")
    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    delays_s = delays_s + float(common_delay_s)
    delay_samples = np.round(delays_s * fs).astype(np.int64)

    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    max_delay_samples = int(np.max(delay_samples))
    n_total_samples = n_signal_samples + max_delay_samples
    print(n_total_samples,"n total samples")
    # Number of output chunks
    n_chunks = (n_total_samples + chunk_size - 1) // chunk_size

    # Doppler schedule
    if doppler_hz_chunks is None:
        doppler_hz_chunks = np.zeros((n_sats, n_chunks), dtype=np.float64)
    else:
        doppler_hz_chunks = np.asarray(doppler_hz_chunks, dtype=np.float64)

        if doppler_hz_chunks.ndim != 2:
            raise ValueError("doppler_hz_chunks must be a 2D array")

        # Accept either (n_sats, n_chunks) or (n_chunks, n_sats)
        if doppler_hz_chunks.shape == (n_sats, n_chunks):
            pass
        elif doppler_hz_chunks.shape == (n_chunks, n_sats):
            doppler_hz_chunks = doppler_hz_chunks.T
        else:
            raise ValueError(
                f"doppler_hz_chunks must have shape ({n_sats}, {n_chunks}) "
                f"or ({n_chunks}, {n_sats}), got {doppler_hz_chunks.shape}"
            )

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Signal samples (undelayed):", n_signal_samples)
    print("Total complex samples:", n_total_samples)
    print("Chunks:", n_chunks)
    print("Doppler schedule shape:", doppler_hz_chunks.shape)

    # Running carrier phase per satellite, so phase stays continuous across chunks
    phase_acc = phase0.astype(np.float64).copy()

    with open(out_file, "wb") as f:
        for chunk_idx, start in enumerate(range(0, n_total_samples, chunk_size)):
            stop = min(start + chunk_size, n_total_samples)
            n_chunk_samples = stop - start

            sample_idx = np.arange(start, stop, dtype=np.int64)

            i_sum = np.zeros(n_chunk_samples, dtype=np.float32)
            q_sum = np.zeros(n_chunk_samples, dtype=np.float32)

            for s in range(n_sats):
                # This chunk's Doppler for satellite s
                f_dopp = doppler_hz_chunks[s, chunk_idx]

                # Delayed source indices
                src_sample_idx = sample_idx - delay_samples[s]
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

                # Build chunk carrier with continuous phase over the whole chunk
                # phase[n] = phase_acc + 2*pi*f_dopp*n/fs
                if n_chunk_samples > 0:
                    n = np.arange(n_chunk_samples, dtype=np.float64)
                    phase_chunk = phase_acc[s] + 2.0 * np.pi * f_dopp * n / fs
                    cos_chunk = np.cos(phase_chunk).astype(np.float32)
                    sin_chunk = np.sin(phase_chunk).astype(np.float32)

                    # Advance stored phase to next chunk boundary
                    phase_acc[s] = (
                        phase_acc[s]
                        + 2.0 * np.pi * f_dopp * n_chunk_samples / fs
                    ) % (2.0 * np.pi)

                # Before the delayed signal begins, contribution stays zero
                if not np.any(valid):
                    continue

                # Sample the chip stream only where valid
                chip_index = np.floor(
                    src_sample_idx[valid].astype(np.float64) * chip_rate / fs
                ).astype(np.int64)
                chip_index = np.clip(chip_index, 0, n_chips - 1)
                baseband = chips[s][chip_index]

                i_sum[valid] += amplitudes[s] * baseband * cos_chunk[valid]
                q_sum[valid] += amplitudes[s] * baseband * sin_chunk[valid]

            iq = np.empty(2 * n_chunk_samples, dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

    return n_total_samples

In [7]:
res3=modulo2_frames_runs(3,Z_count_start,3)
len(res3)

767250000

In [20]:
res3=modulo2_frames_runs(3,Z_count_start,3)#[:100000001]
res6=modulo2_frames_runs(6,Z_count_start,6)#[:100000001]
res9=modulo2_frames_runs(9,Z_count_start,9)#[:100000001]
res16=modulo2_frames_runs(16,Z_count_start,16)#[:100000001]
res21=modulo2_frames_runs(21,Z_count_start,21)#[:100000001]

rho3=np.sqrt(sum((ehpm_to_ECEFlocation(3)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho6=np.sqrt(sum((ehpm_to_ECEFlocation(6)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho9=np.sqrt(sum((ehpm_to_ECEFlocation(9)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho16=np.sqrt(sum((ehpm_to_ECEFlocation(16)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho21=np.sqrt(sum((ehpm_to_ECEFlocation(21)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))

chips_03 = np.frombuffer(res3.unpack(), dtype=np.uint8)
chips_06 = np.frombuffer(res6.unpack(), dtype=np.uint8)
chips_09 = np.frombuffer(res9.unpack(), dtype=np.uint8)
chips_16 = np.frombuffer(res16.unpack(), dtype=np.uint8)
chips_21 = np.frombuffer(res21.unpack(), dtype=np.uint8)

#num_sats = 2

chip_streams_01 = [
    chips_03,
    chips_06,
    chips_09,
    chips_16,
    chips_21,
]

distances_m = np.array([
    rho3,
    rho6,
    rho9,
    rho16,
    rho21,
], dtype=float)

phase0= np.array([
    2*np.pi/1,
    2*np.pi/2,
    2*np.pi/3,
    2*np.pi/4,
    2*np.pi/5,

],dtype=float)


doppler_list = np.array([
    sat_frames["G03"]["D1C_hz"].tolist(),
    sat_frames["G06"]["D1C_hz"].tolist(),
    sat_frames["G09"]["D1C_hz"].tolist(),
    sat_frames["G16"]["D1C_hz"].tolist(),
    sat_frames["G21"]["D1C_hz"].tolist(),
], dtype=float)

delay_list = np.array([
    sat_frames["G03"]["delay_s"].tolist(),
    sat_frames["G06"]["delay_s"].tolist(),
    sat_frames["G09"]["delay_s"].tolist(),
    sat_frames["G16"]["delay_s"].tolist(),
    sat_frames["G21"]["delay_s"].tolist(),
], dtype=float)

pseudorange_list = np.array([
    sat_frames["G03"]["C1C_m"].tolist(),
    sat_frames["G06"]["C1C_m"].tolist(),
    sat_frames["G09"]["C1C_m"].tolist(),
    sat_frames["G16"]["C1C_m"].tolist(),
    sat_frames["G21"]["C1C_m"].tolist(),
], dtype=float)

time_list = sat_frames["G03"]["time"].tolist()

amplitudes = np.array([
    0.1,
    0.1,
    0.1,
    0.1,
    0.1,
],dtype=float)



In [18]:
import numpy as np


def sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01,
    delay_s_chunks,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz_chunks=None,
    phase0=None,
    amplitudes=None,
    noise_std=0.02,
    rng=None,
):
    """
    Generate a summed complex GPS-like IF/baseband signal from multiple PRN chip streams.

    Inputs
    ------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite. All must have same length.
    delay_s_chunks : array
        Time-varying delay per satellite and chunk.
        Shape can be (n_sats, n_chunks) or (n_chunks, n_sats).
    doppler_hz_chunks : array or None
        Time-varying carrier Doppler per satellite and chunk.
        Shape can be (n_sats, n_chunks) or (n_chunks, n_sats).
        If None, zeros are used.
    phase0 : array or None
        Initial carrier phase per satellite, radians.
    amplitudes : array or None
        Amplitude per satellite.
    noise_std : float
        Standard deviation of AWGN added independently to I and Q.
    rng : np.random.Generator or None
        Random number generator.

    Notes
    -----
    - This version uses chunk-varying delay.
    - Carrier Doppler is applied.
    - Code Doppler is approximated from carrier Doppler.
    - Code phase is continuous across chunks via code_phase_acc.
    - Delay is applied as a code-phase shift, not by double-counting absolute sample time.
    """

    L1_HZ = 1575.42e6

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    if rng is None:
        rng = np.random.default_rng()

    n_sats = len(chip_streams_01)

    # Convert chips 0/1 -> -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    n_signal_samples = int(np.floor(signal_duration_s * fs))
    n_total_samples = n_signal_samples

    # Delay schedule
    delay_s_chunks = np.asarray(delay_s_chunks, dtype=np.float64)
    if delay_s_chunks.ndim != 2:
        raise ValueError("delay_s_chunks must be a 2D array")

    if delay_s_chunks.shape[0] == n_sats:
        pass
    elif delay_s_chunks.shape[1] == n_sats:
        delay_s_chunks = delay_s_chunks.T
    else:
        raise ValueError(
            f"delay_s_chunks must have shape (n_sats, n_chunks) "
            f"or (n_chunks, n_sats), got {delay_s_chunks.shape}"
        )

    # Doppler schedule
    if doppler_hz_chunks is None:
        doppler_hz_chunks = np.zeros_like(delay_s_chunks, dtype=np.float64)
    else:
        doppler_hz_chunks = np.asarray(doppler_hz_chunks, dtype=np.float64)

        if doppler_hz_chunks.ndim != 2:
            raise ValueError("doppler_hz_chunks must be a 2D array")

        if doppler_hz_chunks.shape[0] == n_sats:
            pass
        elif doppler_hz_chunks.shape[1] == n_sats:
            doppler_hz_chunks = doppler_hz_chunks.T
        else:
            raise ValueError(
                f"doppler_hz_chunks must have shape (n_sats, n_chunks) "
                f"or (n_chunks, n_sats), got {doppler_hz_chunks.shape}"
            )

    if delay_s_chunks.shape != doppler_hz_chunks.shape:
        raise ValueError(
            f"delay_s_chunks shape {delay_s_chunks.shape} must match "
            f"doppler_hz_chunks shape {doppler_hz_chunks.shape}"
        )

    n_chunks = doppler_hz_chunks.shape[1]
    chunk_size = int(np.ceil(n_signal_samples / n_chunks))

    # Initial carrier phase
    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    # Amplitudes
    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    print("Signal duration [s]:", signal_duration_s)
    print("Signal samples:", n_signal_samples)
    print("Total complex samples:", n_total_samples)
    print("Chunks:", n_chunks)
    print("Chunk size:", chunk_size)
    print("Delay schedule shape:", delay_s_chunks.shape)
    print("Doppler schedule shape:", doppler_hz_chunks.shape)

    # Carrier phase accumulator
    phase_acc = phase0.astype(np.float64).copy()

    # Code phase accumulator in chips
    code_phase_acc = np.zeros(n_sats, dtype=np.float64)

    with open(out_file, "wb") as f:
        for chunk_idx in range(n_chunks):
            start = chunk_idx * chunk_size
            stop = min(start + chunk_size, n_total_samples)
            n_chunk_samples = stop - start

            if n_chunk_samples <= 0:
                break

            local_n = np.arange(n_chunk_samples, dtype=np.float64)

            i_sum = np.zeros(n_chunk_samples, dtype=np.float32)
            q_sum = np.zeros(n_chunk_samples, dtype=np.float32)

            for s in range(n_sats):
                f_dopp = float(doppler_hz_chunks[s, chunk_idx])
                delay_s = float(delay_s_chunks[s, chunk_idx])

                # Carrier phase for this chunk
                phase_chunk = phase_acc[s] + 2.0 * np.pi * f_dopp * local_n / fs
                cos_chunk = np.cos(phase_chunk).astype(np.float32)
                sin_chunk = np.sin(phase_chunk).astype(np.float32)

                phase_acc[s] = (
                    phase_acc[s] + 2.0 * np.pi * f_dopp * n_chunk_samples / fs
                ) % (2.0 * np.pi)

                # Approximate code Doppler
                code_rate = chip_rate * (1.0 + f_dopp / L1_HZ)

                # Convert delay to chips at current code rate
                delay_chips = delay_s * code_rate

                # Continuous code phase in chips for each local sample
                chip_phase_all = (
                    code_phase_acc[s]
                    + local_n * code_rate / fs
                    - delay_chips
                )

                valid = (chip_phase_all >= 0.0) & (chip_phase_all < n_chips)

                if np.any(valid):
                    chip_index = np.floor(chip_phase_all[valid]).astype(np.int64)
                    chip_index = np.clip(chip_index, 0, n_chips - 1)
                    baseband = chips[s][chip_index]

                    i_sum[valid] += amplitudes[s] * baseband * cos_chunk[valid]
                    q_sum[valid] += amplitudes[s] * baseband * sin_chunk[valid]

                # Advance code phase continuously to next chunk
                code_phase_acc[s] = (
                    code_phase_acc[s] + n_chunk_samples * code_rate / fs
                ) % n_chips

            # Add Gaussian noise
            if noise_std > 0:
                i_sum += rng.normal(0.0, noise_std, n_chunk_samples).astype(np.float32)
                q_sum += rng.normal(0.0, noise_std, n_chunk_samples).astype(np.float32)

            iq = np.empty(2 * n_chunk_samples, dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

            print(f"chunk {chunk_idx + 1}/{n_chunks}")

    return n_total_samples



n_total = sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01=chip_streams_01,
    delay_s_chunks=delay_list,
    out_file="gps_signal2_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=2.046e6,
    doppler_hz_chunks=doppler_list,
    phase0=phase0,
    amplitudes=amplitudes,
    noise_std=0.00,
    rng=np.random.default_rng(1234),
)

print("complex samples:", n_total)

Signal duration [s]: 750.0
Signal samples: 1534500000
Total complex samples: 1534500000
Chunks: 751
Chunk size: 2043276
Delay schedule shape: (5, 751)
Doppler schedule shape: (5, 751)
chunk 1/751
chunk 2/751
chunk 3/751
chunk 4/751
chunk 5/751
chunk 6/751
chunk 7/751
chunk 8/751
chunk 9/751
chunk 10/751
chunk 11/751
chunk 12/751
chunk 13/751
chunk 14/751
chunk 15/751
chunk 16/751
chunk 17/751
chunk 18/751
chunk 19/751
chunk 20/751
chunk 21/751
chunk 22/751
chunk 23/751
chunk 24/751
chunk 25/751
chunk 26/751
chunk 27/751
chunk 28/751
chunk 29/751
chunk 30/751
chunk 31/751
chunk 32/751
chunk 33/751
chunk 34/751
chunk 35/751
chunk 36/751
chunk 37/751
chunk 38/751
chunk 39/751
chunk 40/751
chunk 41/751
chunk 42/751
chunk 43/751
chunk 44/751
chunk 45/751
chunk 46/751
chunk 47/751
chunk 48/751
chunk 49/751
chunk 50/751
chunk 51/751
chunk 52/751
chunk 53/751
chunk 54/751
chunk 55/751
chunk 56/751
chunk 57/751
chunk 58/751
chunk 59/751
chunk 60/751
chunk 61/751
chunk 62/751
chunk 63/751
chunk 

In [3]:
print(res3[:100])
print(res6[:100])

NameError: name 'res3' is not defined

In [ ]:
sum(frames(3,Z_count_start))

19546